In [1]:
import numpy as np
import numba
import pandas as pd
import pickle
import operator

# Book Names

In [2]:
bible_books = [
    'Genesis',
    'Exodus',
    'Leviticus',
    'Numbers',
    'Deuteronomy',
    'Joshua',
    'Judges',
    'Ruth',
    '1 Samuel',
    '2 Samuel',
    '1 Kings',
    '2 Kings',
    '1 Chronicles',
    '2 Chronicles',
    'Ezra',
    'Nehemiah',
    'Esther',
    'Job',
    'Psalms',
    'Proverbs',
    'Ecclesiastes',
    'Song of Solomon',
    'Isaiah',
    'Jeremiah',
    'Lamentations',
    'Ezekiel',
    'Daniel',
    'Hosea',
    'Joel',
    'Amos',
    'Obadiah',
    'Jonah',
    'Micah',
    'Nahum',
    'Habakkuk',
    'Zephaniah',
    'Haggai',
    'Zechariah',
    'Malachi',
    'Matthew',
    'Mark',
    'Luke',
    'John',
    'Acts',
    'Romans',
    '1 Corinthians',
    '2 Corinthians',
    'Galatians',
    'Ephesians',
    'Philippians',
    'Colossians',
    '1 Thessalonians',
    '2 Thessalonians',
    '1 Timothy',
    '2 Timothy',
    'Titus',
    'Philemon',
    'Hebrews',
    'James',
    '1 Peter',
    '2 Peter',
    '1 John',
    '2 John',
    '3 John',
    'Jude',
    'Revelation',
]

# Everything else

In [3]:
ot = pd.read_pickle('pickles/sept.pickle')
nt = pd.read_pickle('pickles/tisch.pickle')
strongs = pd.read_pickle('pickles/strongs.pickle')

In [4]:
ot_books = [book for _, book in sorted(ot.groupby('book'), key=operator.itemgetter(0))]
nt_books = [book for _, book in sorted(nt.groupby('book'), key=operator.itemgetter(0))]
all_books = ot_books + nt_books
offsets = [0] * len(all_books)
for book in all_books:
    idx = book.iloc[0]['book']
    if idx == len(offsets):
        print('done')
        break
    offsets[idx] = len(book) + offsets[idx-1]

done


In [5]:
all_book_strongs = [book['str'].values.astype('<U6') for book in all_books]

In [24]:
@numba.njit(parallel=True)
def look_for_references(all_book_strongs, offsets, bible_books):
    by_src = [[(0, 0, 0, 0)]] * len(all_book_strongs)
    for src_id in numba.prange(40, len(all_book_strongs)):
        src_book = all_book_strongs[src_id]
        for dest_id, dest_book in enumerate(all_book_strongs[:39]):
            if src_id == dest_id:
                # Don't look for references from one book to itself
                continue
            for n in range(5, 6):
                for reference_dest_start in range(len(dest_book) - n + 1):
                    for reference_src_start in range(len(src_book) - n + 1):
                        if (dest_book[reference_dest_start:reference_dest_start+n] == src_book[reference_src_start:reference_src_start+n]).all():
                            by_src[src_id].append((reference_src_start+offsets[src_id], n, reference_dest_start+offsets[dest_id], n))
        print('finished finding references in', bible_books[src_id])
    found = []
    for subfound in by_src:
        found.extend(subfound)
    return np.array(found)

In [ ]:
look_for_references(all_book_strongs, offsets, bible_books)